In [ ]:
!pip install -q kagglehub pandas numpy scikit-learn lightgbm catboost matplotlib seaborn optuna nltk

import os
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import nltk
nltk.download('punkt', quiet=True)

from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

import lightgbm as lgb
from catboost import CatBoostClassifier
import optuna

print("Библиотеки установлены и импортированы.")

In [ ]:
import os
import kagglehub
from getpass import getpass

kaggle_dir = os.path.expanduser('~/.kaggle')
os.makedirs(kaggle_dir, exist_ok=True)
token_file = os.path.join(kaggle_dir, 'access_token')

if not os.path.exists(token_file):
    token = getpass("Введите ваш KAGGLE_API_TOKEN (KGAT_...): ")
    with open(token_file, 'w') as f:
        f.write(token.strip())
    os.chmod(token_file, 0o600)
    print("Токен сохранён в ~/.kaggle/access_token")

with open(token_file, 'r') as f:
    os.environ['KAGGLE_API_TOKEN'] = f.read().strip()

competition_path = kagglehub.competition_download('avito-category-prediction')
print(f"Путь к данным: {competition_path}")

train_path = os.path.join(competition_path, 'train.csv')
train_df = pd.read_csv(train_path, nrows=100000)

test_path = os.path.join(competition_path, 'test.csv')
test_df = pd.read_csv(test_path, nrows=20000)

print(f"Размер train: {train_df.shape}")
print(f"Размер test: {test_df.shape}")

In [ ]:
print("="*60)
print("Разведочный анализ данных Avito")
print("="*60)

print("\n[1] Пропуски в данных")
missing = train_df.isnull().sum()
if missing.sum() == 0:
    print("Пропусков нет.")
else:
    print(missing[missing > 0])

print("\n[2] Общая информация")
print(f"• Количество объявлений: {train_df.shape[0]:,}")
print(f"• Количество признаков: {train_df.shape[1]}")
print(f"• Количество категорий: {train_df['Category'].nunique()}")

print("\n[3] Топ-10 самых популярных категорий")
top = train_df['Category_name'].value_counts().head(10)
top_pct = (top / len(train_df) * 100).round(1)
top_df = pd.DataFrame({'Количество': top, 'Доля, %': top_pct})
print(top_df)

fig, ax = plt.subplots(1, 2, figsize=(14, 5))

bars = sns.barplot(x=top.values, y=top.index, ax=ax[0], palette='viridis')
ax[0].set_title('Топ-10 категорий по числу объявлений', fontsize=12)
ax[0].set_xlabel('Количество объявлений')
for i, (val, pct) in enumerate(zip(top.values, top_pct)):
    bars.text(val + 200, i, f'{val} ({pct}%)', va='center')

train_df['desc_len'] = train_df['description'].astype(str).str.len()
sns.histplot(train_df['desc_len'], bins=50, ax=ax[1], color='teal', kde=True)
ax[1].set_title('Распределение длины описания', fontsize=12)
ax[1].set_xlabel('Длина в символах')
ax[1].axvline(train_df['desc_len'].mean(), color='red', linestyle='--', label=f'Среднее: {train_df["desc_len"].mean():.0f}')
ax[1].axvline(train_df['desc_len'].median(), color='orange', linestyle='--', label=f'Медиана: {train_df["desc_len"].median():.0f}')
ax[1].legend()
plt.tight_layout()
plt.show()

print("\n[4] Статистика длины описания")
desc_stats = train_df['desc_len'].describe()
print(desc_stats.round(1))

print("\n[5] Дисбаланс классов")
counts = train_df['Category'].value_counts()
print(f"• Максимальное число объявлений в категории: {counts.max()}")
print(f"• Минимальное: {counts.min()}")
print(f"• Среднее: {counts.mean():.1f}")
print(f"• Медиана: {counts.median():.0f}")

print("\n[6] Самые редкие категории (первые 5)")
rare = train_df['Category_name'].value_counts().tail(5)
print(rare)

train_df.drop(columns=['desc_len'], inplace=True)

In [ ]:
print("="*60)
print("Предобработка данных")
print("="*60)

print("\n[1] Перезагрузка данных для чистоты")
train_path = os.path.join(competition_path, 'train.csv')
test_path = os.path.join(competition_path, 'test.csv')
train_df = pd.read_csv(train_path, nrows=100000)
test_df = pd.read_csv(test_path, nrows=20000)
print(f"Train загружен: {train_df.shape}, Test загружен: {test_df.shape}")

print("\n[2] Объединение title и description")
train_df['text'] = train_df['title'].astype(str) + ' ' + train_df['description'].fillna('').astype(str)
test_df['text'] = test_df['title'].astype(str) + ' ' + test_df['description'].fillna('').astype(str)
print(f"Пример текста: {train_df['text'].iloc[0][:150]}...")

print("\n[3] Отбор нужных колонок")
train_df = train_df[['text', 'Category']]
test_df = test_df[['text']]
print(f"Train shape: {train_df.shape}, Test shape: {test_df.shape}")

print("\n[4] Кодирование целевой переменной")
le = LabelEncoder()
train_df['target'] = le.fit_transform(train_df['Category'])
train_df = train_df[['text', 'target']]
print(f"Количество классов: {len(le.classes_)}")
print(f"Первые 5 классов: {le.classes_[:5].tolist()} -> коды 0..4")

print("\nПример предобработанных данных:")
print(train_df.head())

In [ ]:
print("="*60)
print("Разделение на train/val")
print("="*60)

X = train_df['text']
y = train_df['target']

X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

print(f"Train size: {X_train.shape[0]} ({X_train.shape[0]/len(train_df)*100:.1f}%)")
print(f"Val size: {X_val.shape[0]} ({X_val.shape[0]/len(train_df)*100:.1f}%)")
print(f"Всего: {len(train_df)}")

print("\nРаспределение целевой в train:")
print(y_train.value_counts().head(5))
print("\nРаспределение целевой в val:")
print(y_val.value_counts().head(5))

In [ ]:
print("="*60)
print("Создание числовых признаков из текста")
print("="*60)

import string

def extract_text_features(text_series):
    df = pd.DataFrame()
    df['text_len'] = text_series.str.len()
    df['word_count'] = text_series.str.split().str.len()
    df['unique_words'] = text_series.apply(lambda x: len(set(x.split())))
    df['capital_letters'] = text_series.str.findall(r'[A-ZА-Я]').str.len()
    df['digits'] = text_series.str.count(r'\d')
    df['punctuation'] = text_series.apply(lambda x: sum(1 for ch in x if ch in string.punctuation))
    return df

print("Извлечение признаков из train...")
train_num_features = extract_text_features(X_train)
print(f"Train числовые признаки: {train_num_features.shape}")

print("Извлечение признаков из val...")
val_num_features = extract_text_features(X_val)
print(f"Val числовые признаки: {val_num_features.shape}")

print("\nСтатистика числовых признаков (train):")
print(train_num_features.describe())

print("\nПример первых 5 строк:")
print(train_num_features.head())

In [ ]:
from tqdm import tqdm
from scipy.sparse import vstack
import numpy as np

print("="*60)
print("Векторизация текста с TfidfVectorizer")
print("="*60)

tfidf = TfidfVectorizer(
    max_features=3000,
    ngram_range=(1, 2),
    stop_words='english',
    min_df=5,
    sublinear_tf=True
)

print("Обучение векторизатора на train тексте...")
with tqdm(total=1, desc="Fit") as pbar:
    tfidf.fit(X_train)
    pbar.update(1)

def transform_with_progress(vectorizer, texts, batch_size=5000, desc="Transform"):
    n = len(texts)
    batches = range(0, n, batch_size)
    results = []
    with tqdm(total=len(list(batches)), desc=desc) as pbar:
        for start in batches:
            end = min(start + batch_size, n)
            batch = texts.iloc[start:end]
            transformed = vectorizer.transform(batch)
            results.append(transformed)
            pbar.update(1)
    return vstack(results)

X_train_tfidf = transform_with_progress(tfidf, X_train, desc="Transform train")
X_val_tfidf = transform_with_progress(tfidf, X_val, desc="Transform val")
X_test_tfidf = transform_with_progress(tfidf, test_df['text'], desc="Transform test")

print(f"Train TF-IDF матрица: {X_train_tfidf.shape}")
print(f"Val TF-IDF матрица: {X_val_tfidf.shape}")
print(f"Test TF-IDF матрица: {X_test_tfidf.shape}")
print(f"Количество фич: {len(tfidf.get_feature_names_out())}")

print("\nТоп-10 слов по IDF:")
feature_names = tfidf.get_feature_names_out()
idf_values = tfidf.idf_
top_idx = np.argsort(idf_values)[-10:][::-1]
for idx in top_idx:
    print(f"  {feature_names[idx]}: {idf_values[idx]:.3f}")

In [ ]:
from scipy.sparse import hstack

print("="*60)
print("Объединение TF-IDF с числовыми признаками")
print("="*60)

def extract_text_features(text_series):
    import string
    df = pd.DataFrame({
        'text_len': text_series.str.len(),
        'word_count': text_series.str.split().str.len(),
        'unique_words': text_series.apply(lambda x: len(set(x.split()))),
        'capital_letters': text_series.str.findall(r'[A-ZА-Я]').str.len(),
        'digits': text_series.str.count(r'\d'),
        'punctuation': text_series.apply(lambda x: sum(1 for ch in x if ch in string.punctuation))
    })
    return df

X_train_num = extract_text_features(X_train)
X_val_num = extract_text_features(X_val)
X_test_num = extract_text_features(test_df['text'])

print(f"Train числовые признаки: {X_train_num.shape}")
print(f"Val числовые признаки: {X_val_num.shape}")
print(f"Test числовые признаки: {X_test_num.shape}")

X_train_final = hstack([X_train_tfidf, X_train_num.values])
X_val_final = hstack([X_val_tfidf, X_val_num.values])
X_test_final = hstack([X_test_tfidf, X_test_num.values])

print(f"Финальный train: {X_train_final.shape}")
print(f"Финальный val: {X_val_final.shape}")
print(f"Финальный test: {X_test_final.shape}")

In [ ]:
print("="*60)
print("Обучение Baseline модели (LogisticRegression)")
print("="*60)

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

lr = LogisticRegression(
    multi_class='multinomial',
    max_iter=1000,
    random_state=42,
    solver='lbfgs',
    verbose=1
)

print("Обучение логистической регрессии...")
lr.fit(X_train_final, y_train)

print("Предсказание на валидации...")
y_pred = lr.predict(X_val_final)

acc = accuracy_score(y_val, y_pred)
print(f"\nAccuracy на val: {acc:.4f} ({acc*100:.2f}%)")

print("\nClassification Report (макро-средние):")
print(classification_report(y_val, y_pred, zero_division=0, digits=3))

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

cm = confusion_matrix(y_val, y_pred)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=False, cmap='Blues', fmt='d')
plt.title('Confusion Matrix (Baseline LR)')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.show()

In [ ]:
print("="*60)
print("LightGBM с улучшенными параметрами и метриками")
print("="*60)

from lightgbm import LGBMClassifier
from sklearn.metrics import accuracy_score, classification_report, f1_score, roc_auc_score
from scipy.sparse import hstack
import numpy as np
import pandas as pd
from tqdm import tqdm

try:
    X_train_final_new
except NameError:
    print("Пересоздание TF-IDF с max_features=15000...")
    from sklearn.feature_extraction.text import TfidfVectorizer
    russian_stopwords = ['и', 'в', 'на', 'с', 'по', 'к', 'у', 'о', 'от', 'за', 'из', 'для', 'при', 'без', 'до', 'про',
                         'не', 'да', 'нет', 'бы', 'же', 'ли', 'либо', 'то', 'что', 'как', 'так', 'все', 'это', 'но',
                         'или', 'ибо', 'а', 'ведь', 'вот', 'вон', 'еще', 'уж', 'уже', 'же', 'лишь', 'почти', 'если',
                         'хотя', 'будто', 'словно', 'точно', 'как', 'также', 'тоже', 'ни', 'нибудь', 'что-то', 'кто-то']
    tfidf_new = TfidfVectorizer(max_features=15000, ngram_range=(1,2), stop_words=russian_stopwords, min_df=3, sublinear_tf=True)
    X_train_tfidf_new = tfidf_new.fit_transform(X_train)
    X_val_tfidf_new = tfidf_new.transform(X_val)
    X_test_tfidf_new = tfidf_new.transform(test_df['text'])
    def extract_num_features(text_series):
        import string
        df = pd.DataFrame({
            'text_len': text_series.str.len(),
            'word_count': text_series.str.split().str.len(),
            'unique_words': text_series.apply(lambda x: len(set(x.split()))),
            'capital_letters': text_series.str.findall(r'[A-ZА-Я]').str.len(),
            'digits': text_series.str.count(r'\d'),
            'punctuation': text_series.apply(lambda x: sum(1 for ch in x if ch in string.punctuation))
        })
        return df
    X_train_num = extract_num_features(X_train)
    X_val_num = extract_num_features(X_val)
    X_test_num = extract_num_features(test_df['text'])
    X_train_final_new = hstack([X_train_tfidf_new, X_train_num.values])
    X_val_final_new = hstack([X_val_tfidf_new, X_val_num.values])
    X_test_final_new = hstack([X_test_tfidf_new, X_test_num.values])
    print(f"Матрицы созданы: train {X_train_final_new.shape}, val {X_val_final_new.shape}")

class TqdmCallback:
    def __init__(self):
        self.pbar = None
        self.total = None
    def __call__(self, env):
        if self.pbar is None:
            self.total = env.params.get('num_iterations', 500)
            self.pbar = tqdm(total=self.total, desc="LightGBM", unit='iter', leave=True)
        self.pbar.update(1)
        if env.iteration + 1 >= self.total:
            self.pbar.close()

print("Обучение LightGBM...")
lgb_opt = LGBMClassifier(
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=127,
    max_depth=8,
    min_child_samples=50,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=0.1,
    random_state=42,
    n_jobs=-1,
    verbose=-1
)

lgb_opt.fit(
    X_train_final_new, y_train,
    eval_set=[(X_val_final_new, y_val)],
    eval_metric='multi_logloss',
    callbacks=[TqdmCallback()]
)

print("Предсказание на валидации...")
y_pred_lgb_opt = lgb_opt.predict(X_val_final_new)
y_proba_lgb_opt = lgb_opt.predict_proba(X_val_final_new)

acc = accuracy_score(y_val, y_pred_lgb_opt)
f1_macro = f1_score(y_val, y_pred_lgb_opt, average='macro')
f1_weighted = f1_score(y_val, y_pred_lgb_opt, average='weighted')

try:
    roc_auc = roc_auc_score(y_val, y_proba_lgb_opt, multi_class='ovr', average='macro')
except Exception as e:
    roc_auc = None
    print(f"ROC-AUC не вычислен: {e}")

print(f"\nAccuracy: {acc:.4f} ({acc*100:.2f}%)")
print(f"F1 macro: {f1_macro:.4f}")
print(f"F1 weighted: {f1_weighted:.4f}")
if roc_auc is not None:
    print(f"ROC-AUC (macro ovr): {roc_auc:.4f}")

print("\nПодробный classification report:")
print(classification_report(y_val, y_pred_lgb_opt, zero_division=0, digits=3))

X_train_final = X_train_final_new
X_val_final = X_val_final_new
X_test_final = X_test_final_new
tfidf = tfidf_new
print("\nМатрицы сохранены для CatBoost и финального пайплайна.")

In [ ]:
print("="*60)
print("Обучение CatBoost (GPU)")
print("="*60)

from catboost import CatBoostClassifier
from sklearn.metrics import accuracy_score, classification_report, f1_score, roc_auc_score
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

try:
    import torch
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
except:
    device = 'cpu'
print(f"Доступен GPU: {device == 'cuda'}")

if device == 'cuda':
    task_type = 'GPU'
    devices = '0'
else:
    task_type = 'CPU'
    devices = None

try:
    X_train_final
except NameError:
    print("Пересоздание матриц...")
    from sklearn.feature_extraction.text import TfidfVectorizer
    from scipy.sparse import hstack
    russian_stopwords = ['и', 'в', 'на', 'с', 'по', 'к', 'у', 'о', 'от', 'за', 'из', 'для', 'при', 'без', 'до', 'про',
                         'не', 'да', 'нет', 'бы', 'же', 'ли', 'либо', 'то', 'что', 'как', 'так', 'все', 'это', 'но',
                         'или', 'ибо', 'а', 'ведь', 'вот', 'вон', 'еще', 'уж', 'уже', 'же', 'лишь', 'почти', 'если',
                         'хотя', 'будто', 'словно', 'точно', 'как', 'также', 'тоже', 'ни', 'нибудь', 'что-то', 'кто-то']
    tfidf_new = TfidfVectorizer(max_features=15000, ngram_range=(1,2), stop_words=russian_stopwords, min_df=3, sublinear_tf=True)
    X_train_tfidf_new = tfidf_new.fit_transform(X_train)
    X_val_tfidf_new = tfidf_new.transform(X_val)
    X_test_tfidf_new = tfidf_new.transform(test_df['text'])
    def extract_num_features(text_series):
        import string
        df = pd.DataFrame({
            'text_len': text_series.str.len(),
            'word_count': text_series.str.split().str.len(),
            'unique_words': text_series.apply(lambda x: len(set(x.split()))),
            'capital_letters': text_series.str.findall(r'[A-ZА-Я]').str.len(),
            'digits': text_series.str.count(r'\d'),
            'punctuation': text_series.apply(lambda x: sum(1 for ch in x if ch in string.punctuation))
        })
        return df
    X_train_num = extract_num_features(X_train)
    X_val_num = extract_num_features(X_val)
    X_test_num = extract_num_features(test_df['text'])
    X_train_final = hstack([X_train_tfidf_new, X_train_num.values])
    X_val_final = hstack([X_val_tfidf_new, X_val_num.values])
    X_test_final = hstack([X_test_tfidf_new, X_test_num.values])
    tfidf = tfidf_new
    print(f"Матрицы: train {X_train_final.shape}, val {X_val_final.shape}")

print("Инициализация CatBoost (GPU)...")
cat_model = CatBoostClassifier(
    iterations=150,
    learning_rate=0.1,
    depth=6,
    random_seed=42,
    task_type=task_type,
    devices='0' if task_type=='GPU' else None,
    early_stopping_rounds=30,
    loss_function='MultiClass',
    thread_count=-1,
    border_count=128,
    verbose=50
)

print("Обучение CatBoost...")
cat_model.fit(
    X_train_final, y_train,
    eval_set=(X_val_final, y_val)
)

print("Предсказание на валидации...")
y_pred_cat = cat_model.predict(X_val_final).astype(int)
y_proba_cat = cat_model.predict_proba(X_val_final)

acc = accuracy_score(y_val, y_pred_cat)
f1_macro = f1_score(y_val, y_pred_cat, average='macro')
f1_weighted = f1_score(y_val, y_pred_cat, average='weighted')
try:
    roc_auc = roc_auc_score(y_val, y_proba_cat, multi_class='ovr', average='macro')
except:
    roc_auc = None

print(f"\nAccuracy: {acc:.4f} ({acc*100:.2f}%)")
print(f"F1 macro: {f1_macro:.4f}")
print(f"F1 weighted: {f1_weighted:.4f}")
if roc_auc:
    print(f"ROC-AUC (macro ovr): {roc_auc:.4f}")

print("\nClassification Report (макро-средние):")
print(classification_report(y_val, y_pred_cat, zero_division=0, digits=3))

print("\nТоп-10 важнейших признаков:")
feature_importance = cat_model.feature_importances_
feature_names = list(tfidf.get_feature_names_out()) + ['text_len', 'word_count', 'unique_words', 'capital_letters', 'digits', 'punctuation']
imp_df = pd.DataFrame({'feature': feature_names, 'importance': feature_importance})
imp_df = imp_df.sort_values('importance', ascending=False).head(10)
print(imp_df.to_string(index=False))

plt.figure(figsize=(10,6))
sns.barplot(x='importance', y='feature', data=imp_df, palette='viridis')
plt.title('Топ-10 важных признаков (CatBoost)')
plt.xlabel('Важность')
plt.tight_layout()
plt.show()